# 11 — NUTS vs Laplace: When Does the Approximation Break?

The Laplace covariance reported throughout the paper assumes the
posterior is locally Gaussian at the MAP.  This is a reasonable bet
for prior-free fits, but the framework's
[`run_nuts_vs_laplace.py`](../dev/paper/runners/run_nuts_vs_laplace.py)
discovered that **when a Gaussian prior is wired into the LM
closure**, Laplace under-counts σ by 1.5–17×.

This notebook walks through the comparison at a methodological
level — explaining what to compute, what to expect, and why the
two disagree.  The actual NUTS sampling (~5 min, RAM-heavy) is
better run via the dedicated runner script
`dev/paper/runners/run_nuts_vs_laplace.py` — Jupyter kernels can
OOM on the combined `pyro` + `pytorch` + `nbformat` overhead.
The numbers below are from the paper-validated runner.


## The setup the paper runs

For the actual run see
[`run_nuts_vs_laplace.py`](../dev/paper/runners/run_nuts_vs_laplace.py).
It reads the paramstest, builds a v2 spec, wires a Gaussian prior of
σ = 100 µm on `L_sd` (the typical survey-instrument precision), then:

1. Runs `autocalibrate_pv` to find the MAP and `fisher_at_map` to
   get the Laplace covariance.
2. Runs NUTS via `inference.hmc.hmc_run` (200 warmup + 500 draws by
   default, target acceptance 0.8) on the **same residual closure**.
3. Compares marginal σ.

The closure passed to both methods includes the Gaussian-prior
residual rows (output of `loss.constraints.gaussian_prior_residual`)
concatenated to the data residual.


In [1]:
# Paper-validated numbers, reproduced from
# data/nuts_vs_laplace.csv (run_nuts_vs_laplace.py output).
laplace_sigmas = {
    'Lsd':   1.6732,        # µm
    'BC_y':  7.5950e-04,    # px
    'BC_z':  7.6218e-04,    # px
    'ty':    7.2854e-04,    # deg
    'tz':    7.2704e-04,    # deg
}
nuts_sigmas = {
    'Lsd':   27.7276,       # µm
    'BC_y':  5.8644e-03,    # px
    'BC_z':  1.1513e-03,    # px
    'ty':    1.4492e-03,    # deg
    'tz':    7.6290e-03,    # deg
}

print(f'{"param":<8s}  {"Laplace":>14s}  {"NUTS":>14s}  {"NUTS/Laplace":>14s}')
print('-' * 60)
for nm in ('Lsd', 'BC_y', 'BC_z', 'ty', 'tz'):
    sL = laplace_sigmas[nm]; sN = nuts_sigmas[nm]
    print(f'  {nm:<8s}  {sL:>14.4e}  {sN:>14.4e}  {sN/sL:>14.2f}x')


param            Laplace            NUTS    NUTS/Laplace
------------------------------------------------------------
  Lsd           1.6732e+00      2.7728e+01           16.57x
  BC_y          7.5950e-04      5.8644e-03            7.72x
  BC_z          7.6218e-04      1.1513e-03            1.51x
  ty            7.2854e-04      1.4492e-03            1.99x
  tz            7.2704e-04      7.6290e-03           10.49x


## To run live NUTS

Either run the dedicated runner directly:

```bash
cd .../packages/midas_calibrate_v2/dev/paper
python runners/run_nuts_vs_laplace.py
```

…or, in this notebook, import + run the same code (warning: NUTS +
pyro on the full Varex residual closure can OOM a Jupyter kernel
on systems with < 16 GB RAM):

```python
from midas_calibrate_v2.inference.hmc import hmc_run, HMCConfig
samples = hmc_run(
    spec, log_lik,
    config=HMCConfig(n_warmup=100, n_samples=150,
                      step_size=0.005, target_accept_prob=0.8),
)
nuts_sigmas = {nm: float(samples[nm].std()) for nm in samples}
```

The `runners/` script writes its results to
`data/nuts_vs_laplace.csv`, which is the source of truth for the
table above.


## What you should see

| Param | Laplace | NUTS | ratio |
|---|---|---|---|
| L_sd | ~1.7 µm | ~28 µm | ~17× |
| BC_y | ~7.6e-4 px | ~5.9e-3 px | ~7× |
| BC_z | ~7.6e-4 px | ~1.2e-3 px | ~1.5× |
| ty   | ~7.3e-4 ° | ~1.4e-3 ° | ~2× |
| tz   | ~7.3e-4 ° | ~7.6e-3 ° | ~10× |

Laplace systematically **under-counts** the marginal σ by factors
of 1.5–17×.  The MAP itself agrees between the two methods; the
disagreement is in the posterior *spread*.

## Why?

The LM minimises the **raw sum of squared residuals**.  Data
residuals are dimensionless strain (~10⁻⁵ per fit) and prior rows
are unit-noise (~10⁻¹).  Without per-row noise normalisation, the
prior over-contributes to the LM cost by a factor ~1/σ_r² ~ 10⁸.

The MAP comes out at a **prior-pull-equilibrium** where the local
Hessian is data-dominated (giving a small Laplace σ), but the
actual joint posterior is broader (giving the larger NUTS σ).

## The fix

Pre-scale prior rows by the empirical σ_r before they enter the LM
cost, so the LM minimum coincides with the proper NLL minimum.
This is identified for the next release.  Until then:

- For prior-free fits → Laplace is correct
- For prior-aware fits → use NUTS for σ
- The MAP is unaffected — it's the σ scale that's biased

## See also

- [`dev/paper/runners/run_nuts_vs_laplace.py`](../dev/paper/runners/run_nuts_vs_laplace.py) — the canonical comparison runner
- §"NUTS vs Laplace under priors" subsection of the paper
- `tab:nuts_vs_laplace` in the paper
